In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving whatsapp_classification_dataset_10000.csv to whatsapp_classification_dataset_10000.csv


In [2]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

df.head()

,name,number,isadmin,timestamp,group_name,message_data,word_count,has_placement,has_exam,has_deadline,has_event,category,sentiment_score,urgency_score
0,Priya,1,1,2026-05-19 09:00:00,AI Club,No classes on Friday due to maintenance,7,0,0,0,0,holiday_notice,0.18,9.8
1,Divya,2,1,2026-05-19 09:07:00,CSE Internal Tests,Mock interview scheduled for today evening,6,1,0,0,0,placement_test,0.13,6.7
2,Priya,3,1,2026-05-19 09:14:00,Final Year Projects,No classes on Friday due to maintenance,7,0,0,0,0,holiday_notice,0.84,7.9
3,Rohit,4,1,2026-05-19 09:21:00,CSE Assignments,CN internal exam on Wednesday,5,0,1,0,0,exam_notification,0.15,5.9
4,Rahul,5,0,2026-05-19 09:28:00,CSE Placement Updates,Wipro interview shortlist released,4,1,0,0,0,placement_test,0.47,5.0


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 10000
Columns: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             10000 non-null  object 
 1   number           10000 non-null  int64  
 2   isadmin          10000 non-null  int64  
 3   timestamp        10000 non-null  object 
 4   group_name       10000 non-null  object 
 5   message_data     10000 non-null  object 
 6   word_count       10000 non-null  int64  
 7   has_placement    10000 non-null  int64  
 8   has_exam         10000 non-null  int64  
 9   has_deadline     10000 non-null  int64  
 10  has_event        10000 non-null  int64  
 11  category         10000 non-null  object 
 12  sentiment_score  10000 non-null  float64
 13  urgency_score    10000 non-null  float64
dtypes: float64(2), int64(7), object(5)
memory usage: 1.1+ MB


In [4]:
print(df.isnull().sum())

name               0
number             0
isadmin            0
timestamp          0
group_name         0
message_data       0
word_count         0
has_placement      0
has_exam           0
has_deadline       0
has_event          0
category           0
sentiment_score    0
urgency_score      0
dtype: int64


In [5]:
X = df["message_data"]

print(X.head())

0       No classes on Friday due to maintenance
1    Mock interview scheduled for today evening
2       No classes on Friday due to maintenance
3                 CN internal exam on Wednesday
4            Wipro interview shortlist released
Name: message_data, dtype: object


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(X)

print(X_tfidf.shape)

(10000, 9839)


In [7]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

kmeans.fit(X_tfidf)

print("K-Means Training Completed")

K-Means Training Completed


In [8]:
clusters = kmeans.labels_

print(clusters[:20])

[3 3 3 0 3 3 0 3 3 3 3 3 3 1 3 3 3 3 3 3]


In [9]:
df["Cluster"] = clusters

df.head()

,name,number,isadmin,timestamp,group_name,message_data,word_count,has_placement,has_exam,has_deadline,has_event,category,sentiment_score,urgency_score,Cluster
0,Priya,1,1,2026-05-19 09:00:00,AI Club,No classes on Friday due to maintenance,7,0,0,0,0,holiday_notice,0.18,9.8,3
1,Divya,2,1,2026-05-19 09:07:00,CSE Internal Tests,Mock interview scheduled for today evening,6,1,0,0,0,placement_test,0.13,6.7,3
2,Priya,3,1,2026-05-19 09:14:00,Final Year Projects,No classes on Friday due to maintenance,7,0,0,0,0,holiday_notice,0.84,7.9,3
3,Rohit,4,1,2026-05-19 09:21:00,CSE Assignments,CN internal exam on Wednesday,5,0,1,0,0,exam_notification,0.15,5.9,0
4,Rahul,5,0,2026-05-19 09:28:00,CSE Placement Updates,Wipro interview shortlist released,4,1,0,0,0,placement_test,0.47,5.0,3


In [10]:
print(df["Cluster"].value_counts())

Cluster
3    2538
0    2497
1    2494
2    2471
Name: count, dtype: int64


In [11]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_tfidf, clusters)

print("Silhouette Score =", score)

Silhouette Score = 0.07672096415206384


In [12]:
for cluster in sorted(df["Cluster"].unique()):
    print(f"\nCluster {cluster}")
    print("-" * 50)

    samples = df[df["Cluster"] == cluster]["message_data"].head(5)

    for msg in samples:
        print(msg)


Cluster 0
--------------------------------------------------
CN internal exam on Wednesday
CN internal exam on Wednesday
Hall tickets released for semester exam
Internal practical exam starts tomorrow
Internal practical exam starts tomorrow

Cluster 1
--------------------------------------------------
DBMS record submission before Friday
OS assignment submission deadline tomorrow
DBMS record submission before Friday
OS assignment submission deadline tomorrow
OS assignment submission deadline tomorrow

Cluster 2
--------------------------------------------------
Placement drive by TCS on Wednesday Ref-251
Placement drive by Cognizant on Friday Ref-255
Placement drive by Cognizant on Friday Ref-265
Placement drive by Infosys on Monday Ref-268
Placement drive by HCL on Monday Ref-269

Cluster 3
--------------------------------------------------
No classes on Friday due to maintenance
Mock interview scheduled for today evening
No classes on Friday due to maintenance
Wipro interview shortl